In [17]:
"""
LDA (Latent Dirichlet Allocation) 토픽 모델링 구현
Opinosis 데이터셋을 활용한 코드
"""

import os, glob, re
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from pycaret.clustering import * # setup, create_model, assign_model

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import warnings
import json

warnings.filterwarnings("ignore")

from utils import preprocessing

In [18]:
def LDATopicModeling(n_topics=6, random_state=23):
    """
    Parameters:
    -----------
    n_topics : int
        추출할 토픽의 개수 (K)
    random_state : int
        재현성을 위한 랜덤 시드
    """
    n_topics = n_topics
    random_state = random_state
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words("english"))

    return lemmatizer, stop_words


# ==========================================
# 방법 2: Scikit-learn을 이용한 LDA
# ==========================================
def train_sklearn_lda(documents, n_topics=6, random_state=23):
    """
    Scikit-learn을 이용한 LDA 모델 학습

    Parameters:
    -----------
    documents : list of str
        원본 문서 리스트

    Returns:
    --------
    lda_model : sklearn.decomposition.LatentDirichletAllocation
        학습된 LDA 모델
    vectorizer : sklearn.feature_extraction.text.CountVectorizer
        벡터라이저
    dtm : sparse matrix
        문서-단어 행렬
    """
    # CountVectorizer로 DTM 생성
    vectorizer = CountVectorizer(
        max_df=0.8,
        min_df=2,
        stop_words="english",
        lowercase=True,
        token_pattern="[a-zA-Z\-][a-zA-Z\-]{2,}",
    )

    dtm = vectorizer.fit_transform(documents)

    # LDA 모델 학습
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=random_state,
        max_iter=50,
        learning_method="online",
        n_jobs=-1,
    )

    lda_model.fit(dtm)

    # Perplexity 출력
    perplexity = lda_model.perplexity(dtm)
    print(f"Scikit-learn LDA Perplexity: {perplexity:.4f}")

    return lda_model, vectorizer, dtm


def get_document_topics_sklearn(lda_model, dtm):
    """
    각 문서의 토픽 분포 추출 (Scikit-learn)

    Returns:
    --------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 행렬 (n_documents x n_topics)
    """
    doc_topic_dist = lda_model.transform(dtm)
    return doc_topic_dist


# ==========================================
# 결과 시각화 및 분석
# ==========================================

def display_topics(model, feature_names=None, n_top_words=10, model_type="sklearn"):
    """
    각 토픽의 주요 단어 출력

    Parameters:
    -----------
    model : LDA model
        학습된 LDA 모델
    feature_names : list
        단어 리스트 (sklearn의 경우)
    n_top_words : int
        출력할 단어 개수
    model_type : str
        'gensim' 또는 'sklearn'
    """
    print(f"\n{'='*60}")
    print(f"주요 토픽 및 키워드 ({model_type.upper()})")
    print(f"{'='*60}\n")

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"토픽 {topic_idx + 1}:")
        print(f"  {', '.join(top_words)}\n")

In [19]:
# ==========================================
# 지도 학습 : LDA 출력을 Feature로 활용
# ==========================================

def supervised_learning_example(doc_topic_dist, labels):
    """
    LDA의 문서-토픽 분포를 Feature로 사용한 분류 

    Parameters:
    -----------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 (LDA 출력)
    labels : numpy.ndarray
        실제 레이블 (예: 감성 레이블, 제품 카테고리)
    """
    print(f"\n{'='*60}")
    print("지도 학습 : LDA Feature + Logistic Regression")
    print(f"{'='*60}\n")

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        doc_topic_dist, labels, test_size=0.2, random_state=42
    )

    # Logistic Regression 학습
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_train, y_train)

    # 예측 및 평가
    y_pred = clf.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    


In [20]:
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1267.3


In [21]:
document_df.head()

,filename,opinion_text,processed_text,word_count
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n but for the m...",accurate part find garmin software provides ac...,543
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and ve...",room overly big clean comfortable bed great sh...,739
2,battery-life_amazon_kindle,After I plugged it in to my USB hub on my com...,plugged usb hub computer charge battery chargi...,876
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\...,short battery life moved gb love ipod except b...,620
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh ...",ghz fsb cpu glossy display cell wh li ion batt...,3244


In [22]:
# ==========================================
# 실행 
# ==========================================
lemmatizer, stop_words = LDATopicModeling(n_topics=5, random_state=42)

print("=" * 60)
print("LDA 토픽 모델링 시작")
print("=" * 60)


LDA 토픽 모델링 시작


In [23]:
# ==========================================
# 방법 2: Scikit-learn LDA
# ==========================================
print("\n[방법 2] Scikit-learn을 이용한 LDA\n")

sklearn_model, vectorizer, dtm = train_sklearn_lda(document_df["processed_text"])
doc_topics_sklearn = get_document_topics_sklearn(sklearn_model, dtm)

feature_names = vectorizer.get_feature_names_out()
display_topics(sklearn_model, feature_names, model_type="sklearn", n_top_words=5)

print("\n문서-토픽 분포 (첫 10개 문서):")
print(doc_topics_sklearn[:10])


[방법 2] Scikit-learn을 이용한 LDA

Scikit-learn LDA Perplexity: 700.4577

주요 토픽 및 키워드 (SKLEARN)

토픽 1:
  location, battery, hotel, life, price

토픽 2:
  room, mileage, comfortable, interior, seat

토픽 3:
  video, direction, sound, voice, speed

토픽 4:
  staff, room, service, hotel, food

토픽 5:
  screen, keyboard, button, size, page

토픽 6:
  free, wine, coffee, morning, hotel


문서-토픽 분포 (첫 10개 문서):
[[4.13956131e-04 4.13303424e-04 9.97935647e-01 4.13035849e-04
  4.12508501e-04 4.11548862e-04]
 [2.84245299e-04 9.44075792e-01 2.84027606e-04 5.47850461e-02
  2.84313866e-04 2.86574585e-04]
 [5.39075953e-01 2.58806861e-04 2.60580956e-04 2.58298035e-04
  4.59888382e-01 2.57978730e-04]
 [4.86370571e-01 3.71812527e-04 5.12143329e-01 3.70547470e-04
  3.73597918e-04 3.70142047e-04]
 [6.16154525e-01 6.87861347e-05 5.31189695e-03 6.86349616e-05
  3.78327434e-01 6.87229791e-05]
 [1.44293530e-04 1.44308628e-04 1.44277191e-04 1.43827843e-04
  9.99279634e-01 1.43658452e-04]
 [1.55030814e-04 9.99226013e-01 1.55

In [29]:
def perform_advanced_pycaret_clustering(
    doc_topic_dist,
    documents=None,
    lda_model=None,
    feature_names=None,
    n_top_words=10,
    save_dir="../results"
):
    """
    PyCaret 기반 고급 클러스터링
    + 클러스터별 주요 토픽 출력
    + 결과 콘솔 출력
    + 파일 저장(csv, txt)
    """

    import os
    os.makedirs(save_dir, exist_ok=True)

    print("\n" + "=" * 60)
    print("고급 클러스터링 분석 - 여러 알고리즘 비교")
    print("=" * 60 + "\n")

    # 📌 데이터프레임 구성
    topic_columns = [f"Topic_{i+1}" for i in range(doc_topic_dist.shape[1])]
    df = pd.DataFrame(doc_topic_dist, columns=topic_columns)
    df["Document_ID"] = range(len(df))

    if documents is not None:
        df["Document_Text"] = documents

    cluster_setup = setup(
        data=df,
        session_id=42,
        normalize=True,
        transformation=False,
        ignore_features=["Document_ID"],
        verbose=False,
        html=False,
    )

    algorithms = ["kmeans", "ap", "meanshift", "sc", "hclust", "dbscan"]
    results = {}

    summary_rows = []          # CSV 저장용
    topic_summary_lines = []   # TXT 저장용

    for algo in algorithms:
        try:
            print(f"\n========== {algo.upper()} ==========")

            # 모델 생성
            if algo in ["kmeans", "sc", "hclust"]:
                model = create_model(algo, num_clusters=3)
            else:
                model = create_model(algo)

            predictions = assign_model(model)
            cluster_labels = predictions["Cluster"].values

            # 📌 평가 지표 계산
            from sklearn.metrics import silhouette_score, davies_bouldin_score

            silhouette = silhouette_score(doc_topic_dist, cluster_labels)
            davies_bouldin = davies_bouldin_score(doc_topic_dist, cluster_labels)
            n_clusters = len(np.unique(cluster_labels))

            results[algo] = {
                "model": model,
                "predictions": predictions,
                "silhouette_score": silhouette,
                "davies_bouldin_score": davies_bouldin,
                "n_clusters": n_clusters,
            }

            # 콘솔 출력 + 저장용 데이터 구성
            print(f"Clusters: {n_clusters}")
            print(f"Silhouette Score: {silhouette:.4f}")
            print(f"Davies-Bouldin Score: {davies_bouldin:.4f}")

            summary_rows.append({
                "Algorithm": algo.upper(),
                "N_Clusters": n_clusters,
                "Silhouette": silhouette,
                "Davies_Bouldin": davies_bouldin
            })

            # ======================================================
            # 🔥 클러스터별 주요 토픽 + 키워드 출력 (콘솔 + 파일 저장)
            # ======================================================
            cluster_df = pd.DataFrame(doc_topic_dist)
            cluster_df["cluster"] = cluster_labels
            cluster_topic_mean = cluster_df.groupby("cluster").mean()

            print("\n📌 클러스터별 주요 토픽 및 키워드")
            topic_summary_lines.append(f"\n===== {algo.upper()} =====")

            for c in cluster_topic_mean.index:
                top_topic_idx = cluster_topic_mean.loc[c].idxmax()
                top_topic_num = int(top_topic_idx) + 1

                print(f"- 클러스터 {c}: Top Topic {top_topic_num}")
                topic_summary_lines.append(f"Cluster {c}: Top Topic {top_topic_num}")

                if lda_model is not None:
                    topic_vector = lda_model.components_[top_topic_idx]
                    top_indices = topic_vector.argsort()[-n_top_words:][::-1]
                    top_words = [feature_names[i] for i in top_indices]

                    print(f"    키워드 → {', '.join(top_words)}")
                    topic_summary_lines.append(f"    Keywords: {', '.join(top_words)}")

            # 📁 문서별 클러스터 저장
            assign_path = os.path.join(save_dir, f"cluster_assign_{algo}.csv")
            predictions.to_csv(assign_path, index=False, encoding="utf-8-sig")

        except Exception as e:
            print(f"{algo} 실패: {e}")
            continue

    # ======================================================
    # 📁 성능 요약 CSV 저장
    # ======================================================
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(save_dir, "clustering_summary.csv")
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"\n📁 성능 요약 저장 → {summary_path}")

    # ======================================================
    # 📁 클러스터별 주요 토픽 + 키워드 TXT 저장
    # ======================================================
    topic_summary_path = os.path.join(save_dir, "cluster_topic_summary.txt")
    with open(topic_summary_path, "w", encoding="utf-8") as f:
        f.write("\n".join(topic_summary_lines))
    print(f"📁 주요 토픽 요약 저장 → {topic_summary_path}")

    return results


In [30]:
results = perform_advanced_pycaret_clustering(
    doc_topic_dist=doc_topics_sklearn,
    documents=document_df["opinion_text"],
    lda_model=sklearn_model,
    feature_names=feature_names,
    n_top_words=10,
    save_dir="../results"
)



고급 클러스터링 분석 - 여러 알고리즘 비교


========== KMEANS ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.0215             1.8889          3.7346            0           0   

   Completeness  
0             0  
Clusters: 3
Silhouette Score: 0.3543
Davies-Bouldin Score: 1.1834

📌 클러스터별 주요 토픽 및 키워드
- 클러스터 Cluster 0: Top Topic 5
    키워드 → screen, keyboard, button, size, page, feature, faster, kindle, battery, eye
- 클러스터 Cluster 1: Top Topic 2
    키워드 → room, mileage, comfortable, interior, seat, gas, car, clean, bathroom, transmission
- 클러스터 Cluster 2: Top Topic 1
    키워드 → location, battery, hotel, life, price, service, room, tube, wharf, hour

========== AP ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0       0.042             2.0723          2.6654            0           0   

   Completeness  
0             0  


Clusters: 6
Silhouette Score: 0.7567
Davies-Bouldin Score: 0.4122

📌 클러스터별 주요 토픽 및 키워드
- 클러스터 Cluster 0: Top Topic 2
    키워드 → room, mileage, comfortable, interior, seat, gas, car, clean, bathroom, transmission
- 클러스터 Cluster 1: Top Topic 1
    키워드 → location, battery, hotel, life, price, service, room, tube, wharf, hour
- 클러스터 Cluster 2: Top Topic 4
    키워드 → staff, room, service, hotel, food, friendly, helpful, parking, clean, desk
- 클러스터 Cluster 3: Top Topic 6
    키워드 → free, wine, coffee, morning, hotel, internet, reception, biscotti, hour, evening
- 클러스터 Cluster 4: Top Topic 5
    키워드 → screen, keyboard, button, size, page, feature, faster, kindle, battery, eye
- 클러스터 Cluster 5: Top Topic 3
    키워드 → video, direction, sound, voice, speed, camera, price, map, turn, screen

========== MEANSHIFT ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0           0                  0               0            0           0   

   Completeness  
0             0  
meanshift 실패: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

========== SC ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.0262             2.1002          3.9561            0           0   

   Completeness  
0             0  
Clusters: 3
Silhouette Score: 0.5270
Davies-Bouldin Score: 1.0152

📌 클러스터별 주요 토픽 및 키워드
- 클러스터 Cluster 0: Top Topic 2
    키워드 → room, mileage, comfortable, interior, seat, gas, car, clean, bathroom, transmission
- 클러스터 Cluster 1: Top Topic 3
    키워드 → video, direction, sound, voice, speed, camera, price, map, turn, screen
- 클러스터 Cluster 2: Top Topic 5
    키워드 → screen, keyboard, button, size, page, feature, faster, kindle, battery, eye

========== HCLUST ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.0257             2.0918          3.9803            0           0   

   Completeness  
0             0  


Clusters: 3
Silhouette Score: 0.5141
Davies-Bouldin Score: 1.0342

📌 클러스터별 주요 토픽 및 키워드
- 클러스터 Cluster 0: Top Topic 2
    키워드 → room, mileage, comfortable, interior, seat, gas, car, clean, bathroom, transmission
- 클러스터 Cluster 1: Top Topic 5
    키워드 → screen, keyboard, button, size, page, feature, faster, kindle, battery, eye
- 클러스터 Cluster 2: Top Topic 3
    키워드 → video, direction, sound, voice, speed, camera, price, map, turn, screen

========== DBSCAN ==========


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0           0                  0               0            0           0   

   Completeness  
0             0  


dbscan 실패: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

📁 성능 요약 저장 → ../results\clustering_summary.csv
📁 주요 토픽 요약 저장 → ../results\cluster_topic_summary.txt
